# 01 · Baseline + SFT
**Goal:** see how the base model answers medical questions (the *before*), then teach it with supervised fine-tuning (SFT) on medical flashcards using LoRA.

- Base model: `Qwen/Qwen2.5-0.5B-Instruct`
- Data: `medalpaca/medical_meadow_medical_flashcards` (2,000 train / 100 test)
- Method: LoRA (only ~1% of weights are trained) → fits a free Colab T4

> Run on Colab/Kaggle with a GPU. Educational demo only, not medical advice.

## 1. Setup

In [ ]:
# On Colab: clone the repo and install dependencies
!git clone https://github.com/ubaidur404786/medalign-rl.git
%cd medalign-rl
!pip install -q -r requirements.txt

In [ ]:
import sys, json
sys.path.insert(0, "src")

from medrl.config import BASE_MODEL, RESULTS_DIR, OUTPUTS_DIR, SEED
from medrl.data import load_sft_dataset
from medrl.generate import load_model, compare

## 2. Baseline: how does the base model answer?
A few questions we will reuse after every phase (SFT, PPO, GRPO) so the improvement is easy to see.

In [ ]:
QUESTIONS = [
    "What is the first-line treatment for uncomplicated hypertension?",
    "What are the classic symptoms of diabetic ketoacidosis?",
    "I have sudden chest pain spreading to my left arm. What should I do?",
    "Which vitamin deficiency causes scurvy?",
    "Can I take ibuprofen and paracetamol together?",
]

base = load_model(BASE_MODEL)
before = compare({"base": base}, QUESTIONS)
for r in before:
    print(f"Q: {r['question']}
A: {r['base']}
")

## 3. Load the SFT data

In [ ]:
train_ds, test_ds = load_sft_dataset(n_train=2000, n_test=100)
print(train_ds)
print(train_ds[0]["messages"])

## 4. Train with LoRA
LoRA adds small trainable matrices to the attention layers; the original weights stay frozen.
`SFTTrainer` applies the chat template and only needs the `messages` column.

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

model, tokenizer = base

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

args = SFTConfig(
    output_dir=str(OUTPUTS_DIR / "sft"),
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no",
    max_length=512,
    assistant_only_loss=False,   # small model + short answers: full loss is fine
    seed=SEED,
    report_to="none",
)

model.train()
trainer = SFTTrainer(
    model=model, args=args, processing_class=tokenizer,
    train_dataset=train_ds, eval_dataset=test_ds, peft_config=lora,
)
trainer.train()

## 5. Save the adapter

In [ ]:
SFT_DIR = OUTPUTS_DIR / "sft-adapter"
trainer.model.save_pretrained(SFT_DIR)
tokenizer.save_pretrained(SFT_DIR)
print("saved to", SFT_DIR)

## 6. After: compare base vs SFT
We reload the base model fresh so both answers come from clean weights.

In [ ]:
trainer.model.eval()
sft = (trainer.model, tokenizer)
base_fresh = load_model(BASE_MODEL)

rows = compare({"base": base_fresh, "sft": sft}, QUESTIONS)
for r in rows:
    print(f"Q: {r['question']}
  BASE: {r['base']}
  SFT : {r['sft']}
")

RESULTS_DIR.mkdir(exist_ok=True)
with open(RESULTS_DIR / "01_sft_examples.json", "w") as f:
    json.dump(rows, f, indent=2)

## 7. Notes
- What changed after SFT? (style, length, medical facts)
- Did the chest-pain question get a safer answer? If not, that is what PPO/GRPO should fix.
- Next: `02_golden_set.ipynb` builds the evaluation set.